In [0]:
%sql
CREATE TABLE de_learning_workspace.gold.dim_customer
(
    customer_sk BIGINT GENERATED ALWAYS AS IDENTITY,

    customer_id INT,

    first_name STRING,
    last_name STRING,
    email STRING,
    phone STRING,

    city STRING,
    state STRING,
    country STRING,

    effective_start_date TIMESTAMP,
    effective_end_date TIMESTAMP,

    is_current BOOLEAN,

    created_timestamp TIMESTAMP,
    updated_timestamp TIMESTAMP
)
USING DELTA;

In [0]:
from pyspark.sql.functions import (
    current_timestamp,
    lit
)

silver_customers_df = spark.table(
    "de_learning_workspace.silver.customers"
)


dim_customer_df = (
    silver_customers_df
    .withColumn(
        "effective_start_date",
        current_timestamp()
    )
    .withColumn(
        "effective_end_date",
        lit(None).cast("timestamp")
    )
    .withColumn(
        "is_current",
        lit(True)
    )
    .withColumn(
        "created_timestamp",
        current_timestamp()
    )
    .withColumn(
        "updated_timestamp",
        current_timestamp()
    )
)

In [0]:
dim_customer_df = dim_customer_df.select(
    "customer_id",
    "first_name",
    "last_name",
    "email",
    "phone",
    "city",
    "state",
    "country",
    "effective_start_date",
    "effective_end_date",
    "is_current",
    "created_timestamp",
    "updated_timestamp"
)

In [0]:
(
    dim_customer_df
    .write
    .format("delta")
    .mode("append")
    .saveAsTable(
        "de_learning_workspace.gold.dim_customer"
    )
)

In [0]:
%sql
select * from de_learning_workspace.gold.dim_customer

In [0]:
%sql
update de_learning_workspace.gold.dim_customer as target

set
target.is_current=False , target.effective_end_date=current_timestamp()
where target.is_current=true
and

exists(select 1 from de_learning_workspace.silver.customers as source
where source.customer_id=target.customer_id  and 

(source.city<>target.city or source.state<>target.state or source.country<>target.country))

In [0]:
%sql
INSERT INTO de_learning_workspace.gold.dim_customer
(
    customer_id,
    first_name,
    last_name,
    email,
    phone,
    city,
    state,
    country,
    -- effective_start_date,
    -- effective_end_date,
    -- is_current,
    created_timestamp,
    updated_timestamp
)

select
    customer_id,
    first_name,
    last_name,
    email,
    phone,
    city,
    state,
    country,
    created_at,
    updated_at

from de_learning_workspace.silver.customers as s where not EXISTS (select 1 from de_learning_workspace.gold.dim_customer t where t.customer_id<>s.customer_id
)